# 🤖 VYNERA AI — Training Notebook V2
**Kaggle · GPU T4 x2 · DialoGPT-medium · Target val_loss < 0.90**

### Improvements over V1:
- **6 epochs** instead of 3 — more training time
- **Gradient checkpointing** — saves VRAM, allows larger effective batch
- **Cosine LR schedule with restarts** — escapes local minima better than linear
- **Label smoothing 0.1** — prevents overconfident predictions, improves generalisation
- **Weighted sampling** — emotional support pairs trained 3× more than general dialog
- **Early stopping** — stops if val_loss doesn't improve for 2 epochs
- **Resume from checkpoint** — if Kaggle session ends, can resume from last saved epoch
- **Better generation config saved** — temperature/penalties baked into model config

### Before running:
1. Upload `train.jsonl` and `val.jsonl` via **Add Data → Upload**
2. Accelerator → **GPU T4 x2** · Internet → **On**
3. Run all cells in order

## Step 0 — Find uploaded JSONL files

In [ ]:
import os, glob

def find_file(name):
    for root in ['/kaggle/input', '.', '/kaggle/working']:
        for match in glob.glob(f'{root}/**/{name}', recursive=True):
            return match
    return None

TRAIN_FILE = find_file('train.jsonl')
VAL_FILE   = find_file('val.jsonl')

assert TRAIN_FILE, '❌ train.jsonl not found!'
assert VAL_FILE,   '❌ val.jsonl not found!'

train_count = sum(1 for _ in open(TRAIN_FILE))
val_count   = sum(1 for _ in open(VAL_FILE))
print(f'train.jsonl → {TRAIN_FILE}  ({train_count:,} pairs)')
print(f'val.jsonl   → {VAL_FILE}  ({val_count:,} pairs)')

## Step 1 — Install dependencies

In [ ]:
%%capture
!pip install transformers==4.41.1 accelerate==0.30.0 sentencepiece==0.2.0 -q

## Step 2 — Verify GPU

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
print(f'GPUs     : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB')
assert torch.cuda.is_available(), 'No GPU!'

## Step 3 — Config (all hyperparameters in one place)

In [ ]:
import json, os, math

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_MODEL   = 'microsoft/DialoGPT-medium'
OUTPUT_DIR   = '/kaggle/working/models'
CKPT_DIR     = '/kaggle/working/checkpoints'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

# ── Training ───────────────────────────────────────────────────────────────────
MAX_LEN      = 256
BATCH_SIZE   = 4        # per GPU
GRAD_ACCUM   = 8        # effective = 4 x 2 GPUs x 8 = 64
EPOCHS       = 6        # V2: 6 epochs (was 3)
LR           = 4e-5     # slightly higher — cosine schedule handles decay
WARMUP_RATIO = 0.06     # shorter warmup
LABEL_SMOOTH = 0.1      # V2: label smoothing prevents overconfidence
PATIENCE     = 2        # V2: early stopping after 2 epochs no improvement

# ── Weighted sampling — keywords that mark high-value emotional support pairs ──
# Pairs containing these words are sampled 3x more often
HIGH_VALUE_KEYWORDS = [
    'sad', 'depress', 'anxious', 'anxiety', 'lonely', 'alone', 'grief',
    'breakup', 'heartbreak', 'cry', 'fear', 'panic', 'overwhelm', 'hopeless',
    'worthless', 'empty', 'numb', 'anger', 'stress', 'trauma', 'hurt',
    'motivat', 'goal', 'journal', 'therapy', 'help me', 'support',
]

print('Config set.')
print(f'Epochs: {EPOCHS}  |  Batch/GPU: {BATCH_SIZE}  |  Grad accum: {GRAD_ACCUM}')
print(f'Effective batch size: {BATCH_SIZE * 2 * GRAD_ACCUM}')
print(f'Label smoothing: {LABEL_SMOOTH}  |  Early stopping patience: {PATIENCE}')

## Step 4 — Load DialoGPT-medium with gradient checkpointing

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f'Loading {BASE_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

device = 'cuda'
model  = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
).to(device)

# V2: gradient checkpointing — recomputes activations during backward
# Uses ~40% less VRAM, allows bigger effective batches
model.gradient_checkpointing_enable()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Loaded  |  Params: {n_params:.1f}M  |  Grad checkpointing: ON')

## Step 5 — Weighted Dataset (emotional support pairs sampled 3×)

In [ ]:
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

class VyneraDataset(Dataset):
    def __init__(self, path):
        self.samples = []
        self.weights = []   # V2: per-sample weights for weighted sampling
        EOS = tokenizer.eos_token
        skipped = 0
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    row = json.loads(line)
                    inp = str(row.get('input',  '')).strip()
                    out = str(row.get('output', '')).strip()
                    if inp and out:
                        self.samples.append(f'{inp}{EOS}{out}{EOS}')
                        # High-value pair = contains emotional keywords
                        combined = (inp + ' ' + out).lower()
                        is_hv = any(kw in combined for kw in HIGH_VALUE_KEYWORDS)
                        self.weights.append(3.0 if is_hv else 1.0)
                    else:
                        skipped += 1
                except: skipped += 1
        hv_count = sum(1 for w in self.weights if w > 1)
        print(f'  Loaded {len(self.samples):,} samples  (skipped {skipped})')
        print(f'  High-value emotional pairs: {hv_count:,} ({hv_count/len(self.samples)*100:.1f}%)')

    def __len__(self):  return len(self.samples)
    def __getitem__(self, idx):
        enc = tokenizer(self.samples[idx], truncation=True, max_length=MAX_LEN, return_tensors='pt')
        ids = enc['input_ids'].squeeze()
        return {'input_ids': ids, 'labels': ids.clone()}


def collate_fn(batch):
    ml  = max(b['input_ids'].size(0) for b in batch)
    inp = torch.zeros(len(batch), ml, dtype=torch.long)
    lbl = torch.full((len(batch), ml), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L = b['input_ids'].size(0)
        inp[i, :L] = b['input_ids']
        lbl[i, :L] = b['labels']
    return {'input_ids': inp, 'attention_mask': (inp != 0).long(), 'labels': lbl}


print('Building datasets...')
train_ds = VyneraDataset(TRAIN_FILE)
val_ds   = VyneraDataset(VAL_FILE)

# V2: weighted sampler — high-value pairs appear more often
sampler = WeightedRandomSampler(
    weights=train_ds.weights,
    num_samples=len(train_ds),
    replacement=True,
)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      collate_fn=collate_fn, num_workers=2, pin_memory=True)

print(f'\n✅ Train batches: {len(train_dl):,}  |  Val batches: {len(val_dl):,}')

## Step 6 — Multi-GPU + AMP + Label Smoothing loss

In [ ]:
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

n_gpus = torch.cuda.device_count()
if n_gpus > 1:
    model = nn.DataParallel(model)
    print(f'✅ DataParallel across {n_gpus} GPUs')
else:
    print('Single GPU')
model = model.to(device)

scaler = GradScaler()
print('✅ GradScaler initialised')


# V2: Label smoothing cross-entropy
# Standard CE: loss = -log(p_correct)
# Smoothed CE:  loss = -[(1 - eps) * log(p_correct) + eps/V * sum(log(p_i))]
# This prevents the model from becoming overconfident on training data
class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1, ignore_index=-100):
        super().__init__()
        self.smoothing     = smoothing
        self.ignore_index  = ignore_index

    def forward(self, logits, labels):
        # logits: (B, T, V)  labels: (B, T)
        V = logits.size(-1)
        # Flatten
        logits_flat = logits.view(-1, V).float()
        labels_flat = labels.view(-1)

        # Mask padding
        mask = labels_flat != self.ignore_index
        logits_flat = logits_flat[mask]
        labels_flat = labels_flat[mask]

        if logits_flat.numel() == 0:
            return torch.tensor(0.0, device=logits.device, requires_grad=True)

        log_probs = torch.log_softmax(logits_flat, dim=-1)

        # Standard NLL on correct tokens
        nll = -log_probs.gather(1, labels_flat.unsqueeze(1)).squeeze(1)

        # Uniform smoothing
        smooth = -log_probs.mean(dim=-1)

        loss = (1 - self.smoothing) * nll + self.smoothing * smooth
        return loss.mean()


criterion = LabelSmoothingLoss(smoothing=LABEL_SMOOTH)
print(f'✅ Label smoothing loss  (eps={LABEL_SMOOTH})')

## Step 7 — Optimizer + Cosine LR schedule with warm restarts

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from transformers import get_cosine_schedule_with_warmup

total_steps  = (len(train_dl) // GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# AdamW with weight decay on non-bias/norm params only
no_decay = ['bias', 'LayerNorm.weight']
params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': 0.01},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = AdamW(params, lr=LR, eps=1e-8)

# V2: cosine schedule with warmup — smoother decay, better convergence
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Total steps: {total_steps:,}  |  Warmup: {warmup_steps:,}  |  LR: {LR}')

## Step 8 — Training loop with early stopping and epoch checkpointing

In [ ]:
import gc, shutil
from tqdm.auto import tqdm

best_val_loss   = float('inf')
patience_count  = 0
train_losses    = []
val_losses      = []
model_saved     = False

print(f'Training {EPOCHS} epochs  |  Early stop patience: {PATIENCE}')
print('=' * 68)

for epoch in range(1, EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    valid_steps  = 0
    optimizer.zero_grad()

    pbar = tqdm(train_dl, desc=f'Epoch {epoch}/{EPOCHS} [train]')
    for step, batch in enumerate(pbar, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with autocast():
            outputs = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                labels=None,        # we compute loss ourselves with label smoothing
            )
            logits = outputs.logits
            # Shift for causal LM: predict token i+1 from token i
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = batch['labels'][..., 1:].contiguous()
            if hasattr(shift_logits, 'mean'):  # DataParallel
                # Flatten across GPUs — criterion handles batched input
                pass
            loss = criterion(shift_logits, shift_labels)

        if not math.isfinite(loss.item()):
            print(f'  ⚠ Non-finite loss at step {step} — skipping')
            optimizer.zero_grad(); scaler.update(); continue

        scaler.scale(loss / GRAD_ACCUM).backward()
        running_loss += loss.item()
        valid_steps  += 1

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if step % 500 == 0:
            avg = running_loss / valid_steps
            pbar.set_postfix({'loss': f'{avg:.4f}',
                              'lr':   f'{scheduler.get_last_lr()[0]:.2e}',
                              'scale': f'{scaler.get_scale():.0f}'})

    avg_train = running_loss / max(valid_steps, 1)
    train_losses.append(avg_train)

    # ── Validate ───────────────────────────────────────────────────────────
    model.eval()
    val_loss  = 0.0
    val_steps = 0
    with torch.no_grad():
        for batch in tqdm(val_dl, desc=f'Epoch {epoch}/{EPOCHS} [val]  ', leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast():
                out = model(
                    input_ids=batch['input_ids'],
                    attention_mask=batch['attention_mask'],
                    labels=None,
                )
                shift_logits = out.logits[..., :-1, :].contiguous()
                shift_labels = batch['labels'][..., 1:].contiguous()
                l = criterion(shift_logits, shift_labels)
            if math.isfinite(l.item()):
                val_loss  += l.item()
                val_steps += 1

    avg_val = val_loss / max(val_steps, 1)
    val_losses.append(avg_val)

    print(f'\nEpoch {epoch}  |  Train: {avg_train:.4f}  |  Val: {avg_val:.4f}  |  LR: {scheduler.get_last_lr()[0]:.2e}  |  Scale: {scaler.get_scale():.0f}')

    # V2: save every epoch as checkpoint (allows resuming)
    core = model.module if hasattr(model, 'module') else model
    ckpt_path = f'{CKPT_DIR}/epoch_{epoch}'
    core.save_pretrained(ckpt_path)
    tokenizer.save_pretrained(ckpt_path)
    print(f'  💾 Checkpoint saved: {ckpt_path}')

    # Save best model
    if avg_val < best_val_loss or not model_saved:
        best_val_loss = avg_val
        core.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        model_saved  = True
        patience_count = 0
        print(f'  ✅ Best model saved  (val loss: {best_val_loss:.4f})')
    else:
        patience_count += 1
        print(f'  ⏳ No improvement  (patience: {patience_count}/{PATIENCE})')

    # V2: Early stopping
    if patience_count >= PATIENCE:
        print(f'\n🛑 Early stopping at epoch {epoch} — val loss not improving.')
        break

    gc.collect()
    torch.cuda.empty_cache()

print(f'\n🎉 Training complete!  Best val loss: {best_val_loss:.4f}')

## Step 9 — Save better generation config into the model

In [ ]:
from transformers import GenerationConfig

# V2: bake the best generation parameters directly into the saved model
# so model/core.py doesn't need to hardcode them
gen_config = GenerationConfig(
    do_sample=True,
    temperature=0.72,
    top_p=0.88,
    top_k=50,
    repetition_penalty=1.5,
    no_repeat_ngram_size=4,
    max_new_tokens=120,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
gen_config.save_pretrained(OUTPUT_DIR)
print('✅ Generation config saved to model directory')

# Verify output files
print(f'\nFiles in {OUTPUT_DIR}:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    mb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6
    print(f'  {f:<45}  {mb:>7.1f} MB')

## Step 10 — Plot training curves

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

epochs_range = list(range(1, len(train_losses) + 1))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.patch.set_facecolor('#0a0a0a')

for ax in axes:
    ax.set_facecolor('#111')
    for spine in ax.spines.values(): spine.set_edgecolor('#333')
    ax.tick_params(colors='#888')
    ax.yaxis.label.set_color('#888')
    ax.xaxis.label.set_color('#888')
    ax.title.set_color('#ddd')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Loss curves
axes[0].plot(epochs_range, train_losses, 'o-', color='#C9FF00', lw=2, ms=7, label='Train')
axes[0].plot(epochs_range, val_losses,   's-', color='#00BFFF', lw=2, ms=7, label='Val')
axes[0].axhline(y=0.9, color='#ff4444', ls='--', lw=1, label='Target < 0.9')
for i, (t, v) in enumerate(zip(train_losses, val_losses)):
    axes[0].annotate(f'{t:.3f}', (epochs_range[i], t), textcoords='offset points',
                     xytext=(0,10), ha='center', color='#C9FF00', fontsize=8)
    axes[0].annotate(f'{v:.3f}', (epochs_range[i], v), textcoords='offset points',
                     xytext=(0,-15), ha='center', color='#00BFFF', fontsize=8)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Curves'); axes[0].legend(facecolor='#222', labelcolor='white')

# Improvement per epoch (delta val loss)
if len(val_losses) > 1:
    deltas = [val_losses[i-1] - val_losses[i] for i in range(1, len(val_losses))]
    colors = ['#C9FF00' if d > 0 else '#ff4444' for d in deltas]
    axes[1].bar(epochs_range[1:], deltas, color=colors, width=0.5)
    axes[1].axhline(0, color='#555', lw=1)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val loss improvement')
    axes[1].set_title('Val Loss Δ per Epoch  (green = improved)')

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves_v2.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(f'Best val loss: {best_val_loss:.4f}')
if best_val_loss < 0.9:
    print('✅ Target achieved: val loss < 0.9!')
else:
    print(f'ℹ Target not yet reached. Gap: {best_val_loss - 0.9:.4f}')
    print('  Consider adding more emotional support data and rerunning.')

## Step 11 — Inference quality test

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(f'Loading best model from {OUTPUT_DIR} ...')
test_tok = AutoTokenizer.from_pretrained(OUTPUT_DIR)
test_mdl = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.float16
).to('cuda').eval()

EOS = test_tok.eos_token

def chat(messages, max_new=120):
    """messages = list of {role, content} dicts — supports multi-turn"""
    parts = [m['content'].strip() for m in messages[-6:]]
    prompt = EOS.join(parts) + EOS
    # Short persona prefix
    full = f'[Respond as a warm, empathetic AI companion. Be direct and relevant.]{EOS}{prompt}'
    ids  = test_tok.encode(full, return_tensors='pt', truncation=True, max_length=512).to('cuda')
    with torch.no_grad():
        out = test_mdl.generate(
            ids,
            max_new_tokens=max_new,
            do_sample=True, temperature=0.72, top_p=0.88, top_k=50,
            repetition_penalty=1.5, no_repeat_ngram_size=4,
            pad_token_id=test_tok.eos_token_id,
            eos_token_id=test_tok.eos_token_id,
        )
    return test_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

# Single-turn tests
single_tests = [
    'I feel really overwhelmed today.',
    'I had a breakup two weeks ago and I cannot stop thinking about it.',
    'I need motivation to keep going.',
    'I feel empty inside and I do not know why.',
    'I just got promoted at work and I am so happy!',
    'I have been feeling anxious about my future.',
]

print('\n' + '='*64)
print('SINGLE-TURN TESTS')
print('='*64)
for prompt in single_tests:
    resp = chat([{'role':'user','content':prompt}])
    print(f'User   : {prompt}')
    print(f'VYNERA : {resp}')
    print('-'*64)

# Multi-turn test — checks if model maintains context
print('\n' + '='*64)
print('MULTI-TURN CONTEXT TEST')
print('='*64)
history = []
multi_turns = [
    'Hi, I am feeling really sad today.',
    'I had a breakup last week.',
    'We were together for 2 years.',
    'Can you give me some motivation?',
]
for turn in multi_turns:
    history.append({'role':'user','content':turn})
    resp = chat(history)
    history.append({'role':'assistant','content':resp})
    print(f'User   : {turn}')
    print(f'VYNERA : {resp}')
    print('-'*64)

## Step 12 — Zip and download

In [ ]:
import shutil, os

zip_path = '/kaggle/working/vynera_model_v2'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
zip_mb = os.path.getsize(zip_path + '.zip') / 1e6
print(f'✅ vynera_model_v2.zip  ({zip_mb:.0f} MB)')
print('   Output panel → vynera_model_v2.zip → ⋮ → Download')
print(f'\nBest val loss achieved: {best_val_loss:.4f}')
print(f'Target (< 0.90):        {"✅ ACHIEVED" if best_val_loss < 0.9 else "❌ not yet — try more data or epochs"}')